# 03 — HDBSCAN Parameter Tuning

Grid search over `min_cluster_size` and `cluster_selection_epsilon`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from featrank.ingest.csv_connector import CSVConnector
from featrank.pipeline.cleaner import TextCleaner
from featrank.pipeline.embedder import Embedder
from featrank.pipeline.clusterer import HDBSCANClusterer

sns.set_theme(style='whitegrid')

In [ ]:
requests = CSVConnector('../tests/fixtures/sample_requests.csv').fetch_all()
cleaner = TextCleaner()
kept, cleaned_texts = cleaner.clean_requests(requests)
embedder = Embedder(model_name='all-MiniLM-L6-v2', cache_dir='.cache/minilm')
embeddings = embedder.embed(cleaned_texts)
print(f'Embeddings: {embeddings.shape}')

In [ ]:
min_cluster_sizes = [3, 5, 8, 10, 15]
epsilons = [0.0, 0.1, 0.2, 0.3, 0.4]

rows = []
for mcs in min_cluster_sizes:
    for eps in epsilons:
        c = HDBSCANClusterer(
            min_cluster_size=mcs,
            min_samples=max(1, mcs // 2),
            metric='cosine',
            cluster_selection_epsilon=eps,
        )
        clustered = c.fit_predict(kept, embeddings, cleaned_texts)
        labels = [r.cluster_id for r in clustered]
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = labels.count(-1)
        rows.append({'min_cluster_size': mcs, 'epsilon': eps,
                     'n_clusters': n_clusters, 'noise_pct': round(n_noise/len(labels)*100, 1)})

df = pd.DataFrame(rows)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_clusters = df.pivot('min_cluster_size', 'epsilon', 'n_clusters')
pivot_noise = df.pivot('min_cluster_size', 'epsilon', 'noise_pct')

sns.heatmap(pivot_clusters, annot=True, fmt='d', ax=axes[0], cmap='YlOrRd')
axes[0].set_title('Number of Clusters')

sns.heatmap(pivot_noise, annot=True, fmt='.1f', ax=axes[1], cmap='YlOrRd_r')
axes[1].set_title('Noise %')

plt.suptitle('HDBSCAN Parameter Grid Search', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTarget: 15-25 clusters with <10% noise.')
good = df[(df['n_clusters'].between(15, 25)) & (df['noise_pct'] < 10)]
print(good.to_string())